# Neural Network (NN2) on the MHAR 5-min Feature Set


**Purpose:** The purpose of this notebook is to compute the out-of-sample forecast performance of the `NN2` neural network from Christensen et al. (2023) on the $\mathcal{M}_{\mathrm{HAR}}$ 5-min feature set for the EURO STOXX 50 index over the period 1999--2020.

This implementation follows the paper's neural-network design as closely as is practical in this thesis setup:

- MHAR predictors only: `RVD`, `RVW`, `RVM`
- two hidden layers with `8` and `4` neurons (`NN2` in the paper)
- Leaky ReLU activation with slope `0.01`
- Adam optimizer with learning rate `0.001`
- maximum `500` epochs and early stopping patience `100`
- Glorot (Xavier) normal initialization
- dropout retain probability `0.8` during training
- `400` independent seed initializations ranked by validation MSE
- forecasts from:
  - the single best network (`ensemble size = 1`)
  - the average of the top `5` networks (`ensemble size = 5`)

The neural network is estimated on a fixed train / validation / test split, which is also the approach used for neural networks in the reference paper due to computational cost. Both predictors and the target variable are standardized using training-sample moments before estimation, and forecasts are back-transformed to the original realized-variance scale for evaluation.


## 1. Loading Data
Loading in data from the `MALL_5min.csv` file.


In [33]:
import os
import copy
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

# Load MALL dataset
file_path = "/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/5-min RV/Cleaned Data Sets/MALL_5min.csv"
MALL = pd.read_csv(file_path, index_col="Date", parse_dates=True)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Shape: {MALL.shape}")
print(f"Date range: {MALL.index[0].date()} to {MALL.index[-1].date()}")
print(f"NaN values: {MALL.isna().sum().sum()}")
MALL.head()


Using device: cpu
Shape: (5489, 31)
Date range: 1999-01-05 to 2020-08-06
NaN values: 0


,RVD,RVW,RVM,logRVD,logRVW,logRVM,RVD_pos,RVD_neg,RD_neg,RW_neg,...,BRENT,BRENT_ret,NIKKEI_ret,NIKKEI_sq,SP500_ret_lag,SP500_sq_lag,M1W,M1M,RV_target,logRV_target
Date,,,,,,,,,,,,,,,,,,,,,
1999-01-05,0.000106,0.000089,0.000141,-9.153100,-9.328531,-8.866447,0.000071,0.000035,0.000000,0.000000,...,10.30,-0.060282,-0.013746,0.000189,-0.000920,8.458434e-07,0.087690,0.126206,0.000047,-9.966398
1999-01-06,0.000047,0.000090,0.000132,-9.966398,-9.316927,-8.935760,0.000038,0.000009,0.000000,0.000000,...,10.67,0.035292,0.017657,0.000312,0.013491,1.819949e-04,0.100143,0.187219,0.000237,-8.347779
1999-01-07,0.000237,0.000123,0.000131,-8.347779,-9.004761,-8.938374,0.000110,0.000127,-0.012859,0.000000,...,11.08,0.037706,0.005044,0.000025,0.021899,4.795626e-04,0.078881,0.184020,0.000168,-8.689606
1999-01-08,0.000168,0.000146,0.000114,-8.689606,-8.832472,-9.082830,0.000079,0.000089,-0.005883,0.000000,...,11.70,0.054447,-0.010751,0.000116,-0.002053,4.216638e-06,0.078883,0.165510,0.000282,-8.175339
1999-01-11,0.000282,0.000168,0.000117,-8.175339,-8.692020,-9.056351,0.000067,0.000215,-0.025309,-0.001398,...,12.07,0.031134,-0.001744,0.000003,0.004212,1.774503e-05,0.001055,0.140613,0.000191,-8.565468


## 2. Data Split
### 2.1 Train / Validation / Test Split (70 / 10 / 20)


Following Christensen et al. (2023), the sample is split chronologically into a training set (70%), validation set (10%), and test set (20%). The temporal ordering is strictly preserved to avoid look-ahead bias. For the neural network, this split remains fixed throughout the out-of-sample evaluation.


In [37]:
# Train / Validation / Test split (70 / 10 / 20)
n = len(MALL)
n_train = int(np.floor(0.70 * n))
n_val = int(np.floor(0.10 * n))
n_test = n - n_train - n_val

train_idx = MALL.index[:n_train]
val_idx = MALL.index[n_train:n_train + n_val]
test_idx = MALL.index[n_train + n_val:]

print(f"Total observations: {n}")
print(f"\nTrain:      {train_idx[0].date()} to {train_idx[-1].date()} ({n_train} obs, {n_train/n*100:.1f}%)")
print(f"Validation: {val_idx[0].date()} to {val_idx[-1].date()} ({n_val} obs, {n_val/n*100:.1f}%)")
print(f"Test:       {test_idx[0].date()} to {test_idx[-1].date()} ({n_test} obs, {n_test/n*100:.1f}%)")


Total observations: 5489

Train:      1999-01-05 to 2014-02-26 (3842 obs, 70.0%)
Validation: 2014-02-27 to 2016-04-20 (548 obs, 10.0%)
Test:       2016-04-21 to 2020-08-06 (1099 obs, 20.0%)


## 3. MHAR Feature Set Construction and Standardization

The `NN2` network is estimated on the MHAR feature set only, consisting of the daily, weekly, and monthly lags of realized variance. In line with the paper, the explanatory variables are standardized using training-sample moments. To improve numerical stability in neural-network estimation, the target variable is also standardized using the same training window and later back-transformed for forecast evaluation.


In [11]:
# Define target and MHAR feature set
feature_cols = ["RVD", "RVW", "RVM"]

y = MALL["RV_target"].values
X_MHAR = MALL[feature_cols].values

# Raw splits
X_train = X_MHAR[:n_train]
X_val = X_MHAR[n_train:n_train + n_val]
X_test = X_MHAR[n_train + n_val:]

y_train = y[:n_train]
y_val = y[n_train:n_train + n_val]
y_test = y[n_train + n_val:]

# Feature scaling using training moments only
X_mean = X_train.mean(axis=0)
X_std = X_train.std(axis=0, ddof=0)
X_std = np.where(X_std == 0, 1.0, X_std)

X_train_scaled = (X_train - X_mean) / X_std
X_val_scaled = (X_val - X_mean) / X_std
X_test_scaled = (X_test - X_mean) / X_std

# Target scaling using training moments only
y_mean = y_train.mean()
y_std = y_train.std(ddof=0)
if y_std == 0:
    y_std = 1.0

y_train_scaled = (y_train - y_mean) / y_std
y_val_scaled = (y_val - y_mean) / y_std
y_test_scaled = (y_test - y_mean) / y_std

print("Feature set:")
for col in feature_cols:
    print(f"  - {col}")

print(f"\nTraining set:   X={X_train_scaled.shape}, y={y_train.shape}")
print(f"Validation set: X={X_val_scaled.shape}, y={y_val.shape}")
print(f"Test set:       X={X_test_scaled.shape}, y={y_test.shape}")
print("\nPredictor standardization check:")
print(f"  Means: {X_train_scaled.mean(axis=0).round(6)}")
print(f"  Std:   {X_train_scaled.std(axis=0).round(6)}")
print("\nTarget standardization check:")
print(f"  Train mean (scaled): {y_train_scaled.mean():.6f}")
print(f"  Train std  (scaled): {y_train_scaled.std():.6f}")
print(f"  Raw target mean:     {y_mean:.6e}")
print(f"  Raw target std:      {y_std:.6e}")


Feature set:
  - RVD
  - RVW
  - RVM

Training set:   X=(3842, 3), y=(3842,)
Validation set: X=(548, 3), y=(548,)
Test set:       X=(1099, 3), y=(1099,)

Predictor standardization check:
  Means: [ 0. -0. -0.]
  Std:   [1. 1. 1.]

Target standardization check:
  Train mean (scaled): 0.000000
  Train std  (scaled): 1.000000
  Raw target mean:     1.608657e-04
  Raw target std:      2.992674e-04


## 4. Define the NN2 Architecture

Following the paper, `NN2` is the second neural network in the geometric pyramid family. It contains two hidden layers with `8` and `4` neurons. We use Leaky ReLU with slope `0.01` and Glorot normal initialization. The paper reports dropout `0.8`; here this is implemented as a retain probability of `0.8`, corresponding to a PyTorch dropout probability of `0.2` during training.


In [14]:
class NN2(nn.Module):
    def __init__(self, input_dim, hidden_dims=(8, 4), keep_prob=0.8):
        super().__init__()
        self.hidden1 = nn.Linear(input_dim, hidden_dims[0])
        self.hidden2 = nn.Linear(hidden_dims[0], hidden_dims[1])
        self.activation = nn.LeakyReLU(negative_slope=0.01)
        self.dropout = nn.Dropout(p=1.0 - keep_prob)
        self.output = nn.Linear(hidden_dims[1], 1)
        self._initialize_weights()

    def _initialize_weights(self):
        for layer in [self.hidden1, self.hidden2, self.output]:
            nn.init.xavier_normal_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, x):
        x = self.hidden1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.hidden2(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.output(x)
        return x.squeeze(-1)

input_dim = X_train_scaled.shape[1]
model_test = NN2(input_dim=input_dim).to(device)

print(model_test)
print(f"\nInput dimension:   {input_dim}")
print("Hidden neurons:    8, 4")
print("Output dimension:  1")
print(f"\nTotal parameters: {sum(p.numel() for p in model_test.parameters())}")


NN2(
  (hidden1): Linear(in_features=3, out_features=8, bias=True)
  (hidden2): Linear(in_features=8, out_features=4, bias=True)
  (activation): LeakyReLU(negative_slope=0.01)
  (dropout): Dropout(p=0.19999999999999996, inplace=False)
  (output): Linear(in_features=4, out_features=1, bias=True)
)

Input dimension:   3
Hidden neurons:    8, 4
Output dimension:  1

Total parameters: 73


## 5. Training Function with Early Stopping

The network is trained by minimizing mean squared error in standardized target space using Adam with learning rate `0.001`. Training runs for at most `500` epochs. Early stopping is activated if validation loss does not improve for `100` consecutive epochs, after which the best validation-state parameters are restored.


In [17]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def to_tensor(x):
    return torch.tensor(x, dtype=torch.float32, device=device)


def train_nn2(X_tr, y_tr, X_vl, y_vl,
              input_dim,
              hidden_dims=(8, 4),
              keep_prob=0.8,
              learning_rate=0.001,
              max_epochs=500,
              patience=100,
              seed=0):

    set_seed(seed)

    X_tr_t = to_tensor(X_tr)
    y_tr_t = to_tensor(y_tr)
    X_vl_t = to_tensor(X_vl)
    y_vl_t = to_tensor(y_vl)

    model = NN2(input_dim=input_dim, hidden_dims=hidden_dims, keep_prob=keep_prob).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    best_val_loss = np.inf
    best_state = None
    best_epoch = 0
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad()
        pred_tr = model(X_tr_t)
        loss_tr = criterion(pred_tr, y_tr_t)
        loss_tr.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            pred_vl = model(X_vl_t)
            loss_vl = criterion(pred_vl, y_vl_t).item()

        if loss_vl < best_val_loss:
            best_val_loss = loss_vl
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()

    return model, best_val_loss, best_epoch


## 6. Initial Validation Check

Before estimating the large seed ensemble, we train a single network on the initial training / validation split as a sanity check. The validation loss is reported in standardized target space, while predictions are back-transformed to the original realized-variance scale for interpretation.


In [20]:
model_check, val_loss_check, best_epoch_check = train_nn2(
    X_tr=X_train_scaled,
    y_tr=y_train_scaled,
    X_vl=X_val_scaled,
    y_vl=y_val_scaled,
    input_dim=input_dim,
    seed=0
)

with torch.no_grad():
    preds_check_scaled = model_check(to_tensor(X_val_scaled)).detach().cpu().numpy()
preds_check = preds_check_scaled * y_std + y_mean

print(f"Validation MSE (scaled, seed=0): {val_loss_check:.6e}")
print(f"Best epoch:                      {best_epoch_check}")
print(f"Mean validation forecast:       {preds_check.mean():.6e}")
print(f"Mean actual validation RV:      {y_val.mean():.6e}")
print(f"Any negative forecasts:         {(preds_check < 0).sum()}")
print(f"Any NaN forecasts:              {np.isnan(preds_check).sum()}")


Validation MSE (scaled, seed=0): 1.229610e-01
Best epoch:                      500
Mean validation forecast:       1.290742e-04
Mean actual validation RV:      1.221296e-04
Any negative forecasts:         0
Any NaN forecasts:              0


## 7. Ensemble Training on the Fixed Estimation Window

Following the paper's seed-ranking idea, we train `400` independent versions of the same `NN2` architecture with different random seeds. The networks are ranked by validation MSE in standardized target space. We then retain:

- the single best network (`ensemble size = 1`)
- the top `5` networks (`ensemble size = 5`)


In [ ]:
N_SEEDS = 400
TOP_K = 5

seed_records = []
start_time = time.time()

print(f"Training {N_SEEDS} independent NN2 networks...")
for seed in range(N_SEEDS):
    model_i, val_loss_i, best_epoch_i = train_nn2(
        X_tr=X_train_scaled,
        y_tr=y_train_scaled,
        X_vl=X_val_scaled,
        y_vl=y_val_scaled,
        input_dim=input_dim,
        seed=seed
    )
    seed_records.append({
        "seed": seed,
        "val_loss": val_loss_i,
        "best_epoch": best_epoch_i,
        "model": model_i
    })

    if (seed + 1) % 25 == 0:
        elapsed = (time.time() - start_time) / 60
        print(f"  Completed {seed + 1:3d}/{N_SEEDS} seeds | Elapsed: {elapsed:.2f} min")

elapsed = (time.time() - start_time) / 60
print(f"\nTraining time: {elapsed:.2f} minutes")

# Rank by validation loss
seed_records.sort(key=lambda x: x["val_loss"])
best_network = seed_records[0]
top5_networks = seed_records[:TOP_K]

ranking_df = pd.DataFrame([
    {
        "rank": i + 1,
        "seed": rec["seed"],
        "val_mse_scaled": rec["val_loss"],
        "best_epoch": rec["best_epoch"]
    }
    for i, rec in enumerate(seed_records)
])

print("\nTop seeds by validation MSE:")
display(ranking_df.head(10))


Training 400 independent NN2 networks...
  Completed  25/400 seeds | Elapsed: 0.11 min
  Completed  50/400 seeds | Elapsed: 0.22 min
  Completed  75/400 seeds | Elapsed: 0.33 min
  Completed 100/400 seeds | Elapsed: 0.44 min
  Completed 125/400 seeds | Elapsed: 0.54 min
  Completed 150/400 seeds | Elapsed: 0.65 min
  Completed 175/400 seeds | Elapsed: 0.76 min
  Completed 200/400 seeds | Elapsed: 0.87 min
  Completed 225/400 seeds | Elapsed: 0.98 min
  Completed 250/400 seeds | Elapsed: 1.09 min
  Completed 275/400 seeds | Elapsed: 1.20 min
  Completed 300/400 seeds | Elapsed: 1.31 min
  Completed 325/400 seeds | Elapsed: 1.42 min


## 8. One-Day-Ahead Test Forecasts

The selected `NN2` networks are now used to forecast the entire test set. The forecasts are first produced in standardized target space and then back-transformed to the original realized-variance scale. If any forecast is negative, it is replaced by the minimum realized variance observed in the pre-test sample (training plus validation), which acts as the non-negativity safeguard used elsewhere in the thesis.


In [40]:
X_test_t = to_tensor(X_test_scaled)
min_rv_insample = y[:n_train + n_val].min()

# Best network forecast
best_network["model"].eval()
with torch.no_grad():
    pred_best_scaled = best_network["model"](X_test_t).detach().cpu().numpy()
pred_best = pred_best_scaled * y_std + y_mean
pred_best = np.maximum(pred_best, min_rv_insample)

# Top-5 ensemble forecast
pred_top5_all = []
for rec in top5_networks:
    rec["model"].eval()
    with torch.no_grad():
        pred_i_scaled = rec["model"](X_test_t).detach().cpu().numpy()
        pred_i = pred_i_scaled * y_std + y_mean
        pred_top5_all.append(pred_i)

pred_top5 = np.mean(pred_top5_all, axis=0)
pred_top5 = np.maximum(pred_top5, min_rv_insample)

results_NN2 = pd.DataFrame({
    "RV_actual": y_test,
    "RV_forecast_best": pred_best,
    "RV_forecast_top5": pred_top5
}, index=test_idx)

errors_best = results_NN2["RV_actual"] - results_NN2["RV_forecast_best"]
errors_top5 = results_NN2["RV_actual"] - results_NN2["RV_forecast_top5"]

mse_best = np.mean(errors_best ** 2)
mse_top5 = np.mean(errors_top5 ** 2)

print(f"Forecast period: {test_idx[0].date()} to {test_idx[-1].date()}")
print(f"In-sample floor: {min_rv_insample:.6e}")
print(f"\nNN2_best MSE: {mse_best:.6e}")
print(f"NN2_top5 MSE: {mse_top5:.6e}")


Forecast period: 2016-04-21 to 2020-08-06
In-sample floor: 3.910328e-08

NN2_best MSE: 5.894976e-08
NN2_top5 MSE: 5.620558e-08


## 9. Save Forecasts and Summary Statistics


In [ ]:
forecast_path = "/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/5-min RV/Cleaned Data Sets/Forecasts 5-min/MHAR/Neural Networks"
mse_path = "/Users/tobiasbergdahlpersson/Documents/SSE/MSc Thesis/5-min RV/Cleaned Data Sets/MSE 5-min/MHAR/Neural Networks"

os.makedirs(forecast_path, exist_ok=True)
os.makedirs(mse_path, exist_ok=True)

results_NN2.to_csv(os.path.join(forecast_path, "forecasts_NN2.csv"))
mse_summary_NN2 = pd.DataFrame({
    "Model": ["NN2_best", "NN2_top5"],
    "MSE": [mse_best, mse_top5],
    "RMSE": [np.sqrt(mse_best), np.sqrt(mse_top5)],
    "MAE": [errors_best.abs().mean(), errors_top5.abs().mean()],
    "Mean_Error": [errors_best.mean(), errors_top5.mean()]
}).set_index("Model")

mse_summary_NN2.to_csv(os.path.join(mse_path, "mse_summary_NN2.csv"))

print(f"Forecasts saved to:   {forecast_path}")
print(f"MSE summary saved to: {mse_path}")
print(f"Rows saved: {len(results_NN2)}")
print()
print(mse_summary_NN2.to_string())


## 10. Results
### 10.1 Forecast Visualization


In [ ]:
results_plot = pd.DataFrame({
    "Actual": np.sqrt(results_NN2["RV_actual"] * 252) * 100,
    "Forecast_best": np.sqrt(results_NN2["RV_forecast_best"] * 252) * 100,
    "Forecast_top5": np.sqrt(results_NN2["RV_forecast_top5"] * 252) * 100
}, index=results_NN2.index)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(results_plot["Actual"], label="Actual RV", color="black", linewidth=0.8)
ax.plot(results_plot["Forecast_best"], label="NN2 Best Forecast", color="red", linewidth=0.8, linestyle="--")
ax.plot(results_plot["Forecast_top5"], label="NN2 Top-5 Ensemble", color="blue", linewidth=0.8, linestyle=":")
ax.set_title("NN2 — One-Day-Ahead Realized Volatility Forecast\nEURO STOXX 50 (2016–2020)", fontsize=12)
ax.set_ylabel("Annualized Volatility (%)")
ax.set_xlabel("Date")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nSummary of forecast errors:")
print("\nNN2 Best:")
print(f"  Mean error:  {errors_best.mean():.6e}")
print(f"  RMSE:        {np.sqrt(mse_best):.6e}")
print(f"  MAE:         {errors_best.abs().mean():.6e}")

print("\nNN2 Top-5 Ensemble:")
print(f"  Mean error:  {errors_top5.mean():.6e}")
print(f"  RMSE:        {np.sqrt(mse_top5):.6e}")
print(f"  MAE:         {errors_top5.abs().mean():.6e}")
